Q1

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import shapiro

df = pd.read_csv("/Users/henry/Desktop/5580/project1/data.csv")

results = []
for ticker, g in df.groupby("TICKER"):
    g = g.sort_values("Month")
    log_ret = np.log1p(g["RETC"])   #取该股票120个月的RETC，然后就转成对数收益
    W, p = shapiro(log_ret)#做Shapiro-Wilk
    results.append({"TICKER": ticker, "W": W, "p_value": p, "is_normal_5pct": p > 0.05}) #判定

res = pd.DataFrame(results).sort_values("TICKER")
normal_tickers = res.loc[res["is_normal_5pct"], "TICKER"].tolist()

print("正态股票数量:", len(normal_tickers))
print("正态股票ticker:", normal_tickers)



正态股票数量: 20
正态股票ticker: ['AAPL', 'AMGN', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'INTC', 'JNJ', 'MCD', 'MRK', 'MSFT', 'PG', 'TRV', 'UNH', 'VZ', 'WBA', 'WMT']


Results Interpretation

In [4]:
res

,TICKER,W,p_value,is_normal_5pct
0,AAPL,0.985307,0.218908,True
1,AMGN,0.987426,0.334709,True
2,AXP,0.932658,0.000014,False
3,BA,0.989551,0.495126,True
4,C,0.972024,0.013253,False
5,CAT,0.991459,0.669865,True
6,CRM,0.990905,0.617398,True
7,CSCO,0.982191,0.113739,True
8,CVX,0.989998,0.534116,True
9,DIS,0.987426,0.334679,True


Q2

In [5]:
import pandas as pd
import numpy as np

data = pd.read_csv("/Users/henry/Desktop/5580/project1/data.csv")
rf = pd.read_csv("/Users/henry/Desktop/5580/project1/RF_data.csv")

data["Month"] = data["Month"].astype(int)
rf["Month"] = rf["Month"].astype(int)

rf_map = dict(zip(rf["Month"], rf["RF"]))

def next_month(yyyymm):
    y = yyyymm // 100
    m = yyyymm % 100
    return (y + 1) * 100 + 1 if m == 12 else y * 100 + (m + 1)

#這個月的RF
data["RF_t"] = data["Month"].map(rf_map)

#下個月的rf
data["Month_next"] = data["Month"].apply(next_month)
data["RF_t1"] = data["Month_next"].map(rf_map)

#下月超额收益y
data["y"] = data["RETN"] - data["RF_t1"]

#減掉無風險利率
data["RETC_excess"] = data["RETC"] - data["RF_t"]
data["sprtrn_excess"] = data["sprtrn"] - data["RF_t"]


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV

#切分
train = data[data["Month"] <= 201711].copy()
test  = data[data["Month"] > 201711].copy()

drop_cols = [
    "TICKER", "Month", "RETN", "y",
    "RF_t", "RF_t1", "Month_next",
    "RETC", "sprtrn"
]
X_cols = [c for c in data.columns if c not in drop_cols]

X_train = train[X_cols]
X_test  = test[X_cols]
y_train = train["y"].values
y_test  = test["y"].values

#standardize
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

#訓練模型
# OLS
ols = LinearRegression()
ols.fit(X_train_s, y_train)

#LASSO
lasso = LassoCV(cv=5, random_state=0, max_iter=20000)
lasso.fit(X_train_s, y_train)

#PLS
best_k = 1
best_score = -1e18
for k in range(1, min(20, X_train_s.shape[1]) + 1):
    m = PLSRegression(n_components=k)
    s = cross_val_score(
        m, X_train_s, y_train,
        cv=5, scoring="neg_mean_squared_error"
    ).mean()
    if s > best_score:
        best_score = s
        best_k = k

pls = PLSRegression(n_components=best_k)
pls.fit(X_train_s, y_train)

# Boosted Trees
param_grid = {
    "n_estimators": [100, 300],
    "learning_rate": [0.05, 0.1],
    "max_depth": [1, 2]
}
gb_cv = GridSearchCV(
    GradientBoostingRegressor(random_state=0),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
gb_cv.fit(X_train_s, y_train)
boost = gb_cv.best_estimator_


In [8]:
pred = {
    "OLS": ols.predict(X_test_s),
    "LASSO": lasso.predict(X_test_s),
    "PLS": pls.predict(X_test_s).ravel(),
    "BoostedTrees": boost.predict(X_test_s)
}

r_bar = y_test.mean()
den = np.sum((y_test - r_bar) ** 2)
result = []
for name, y_hat in pred.items():
    num = np.sum((y_test - y_hat) ** 2)
    r2_oos = 1 - num / den
    result.append([name, r2_oos])

result_df = pd.DataFrame(result, columns=["Model", "R2_oos"]).sort_values("R2_oos", ascending=False)

print("Best PLS components:", best_k)
print("Best Boost params:", gb_cv.best_params_)

Best PLS components: 1
Best Boost params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100}


In [9]:
result_df

,Model,R2_oos
1,LASSO,0.014375
2,PLS,-0.019065
0,OLS,-0.033178
3,BoostedTrees,-0.044925


Results Interpretation

Only LASSO has a positive R2_oos, which means that it is slightly better than "predicting using the test period mean". AND PLS,OLS and Boosted trees all have negative values, meaning their prediction performance is not as good as the mean benchmark.

Q3

Method1

In [16]:
from sklearn.linear_model import LassoCV
import pandas as pd

lasso = LassoCV(cv=5, random_state=0, max_iter=20000)
lasso.fit(X_train_s, y_train)

coef = pd.Series(lasso.coef_, index=X_cols)
importance_A = coef.abs().sort_values(ascending=False)

print("Best alpha:", lasso.alpha_)
print("Non-zero coefficients:", (coef != 0).sum(), "/", len(coef))


Best alpha: 0.0011620778342855307
Non-zero coefficients: 18 / 49


In [17]:
print("\nTop 10 by |coef|:")
importance_A.head(10)


Top 10 by |coef|:


sprtrn_excess    0.003234
fcf_ocf          0.002086
divyield         0.001613
pe_op_basic      0.001288
intcov_ratio     0.001256
ivol             0.000890
pcf              0.000829
npm              0.000819
CAPEI            0.000747
RETC_excess      0.000635
dtype: float64

This indicates that under 5-fold cross-validation, LASSO used alpha=0.001162 to screen out 18 effective variables from 49 variables, among which sprtrn_excess, fcf_ocf, and divyield made the largest relative contributions to the prediction of excess returns for the next month.

Method2

In [18]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
import pandas as pd

param_grid = {
    "n_estimators": [100, 300],
    "learning_rate": [0.05, 0.1],
    "max_depth": [1, 2]
}

gb_cv = GridSearchCV(
    GradientBoostingRegressor(random_state=0),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

gb_cv.fit(X_train_s, y_train)
boost = gb_cv.best_estimator_

importance_B = pd.Series(boost.feature_importances_, index=X_cols).sort_values(ascending=False)

print("Best params:", gb_cv.best_params_)

Best params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100}


In [19]:
print("\nTop 10 by feature importance:")
importance_B.head(10)


Top 10 by feature importance:


sprtrn_excess    0.571156
ptb              0.050806
pcf              0.050575
ps               0.038678
pe_op_basic      0.037419
OSC              0.029159
cash_debt        0.027791
pe_exi           0.022591
invt_act         0.017600
sale_nwc         0.014952
dtype: float64

sprtrn_excess contributes significantly more to prediction than other variables, while ptb, pcf, ps, and pe_op_basic are important features in the second tier.

Results Interpretation:
The Top 10 results for both methods share three common variables: sprtrn_excess, pcf and pe_op_basic, which means these three features exhibit relatively stable importance across different models.

Q4

These results show that next month's returns have only weak and unstable predictability: only LASSO's out-of-sample R^2 is slightly positive , while the others are negative.

Therefore, these results broadly support the semi-strong efficient market hypothesis, which states that publicly available information is difficult to predict reliably and yield significant excess returns.